# 🏟️ LaLiga Teams — Data Pipeline (Bronze → Silver)

Notebook ini menyusun seluruh pipeline data untuk **teams** secara interaktif.

**Alur:**
```
datasets/teams/ (Source, 6 JSON)
    ↓  Bronze Ingestion (copy file)
data/bronze/teams/ (Bronze, 6 JSON)
    ↓  Step 1: json_to_csv()
data/silver/teams/csv_raw/ (CSV 0, apa adanya)
    ↓  Step 2: transform()
data/silver/teams/csv_clean/ (CSV 1, bersih)
```

---
## ⚙️ Setup & Config

In [ ]:
import shutil
import json
import logging
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Path Config
# ---------------------------------------------------------------------------
# Karena notebook ada di z_notebook/, kita naik satu level ke root project
BASE_DIR = Path.cwd().parent  # root project

SOURCE_DIR = BASE_DIR / "datasets" / "teams"
BRONZE_DIR = BASE_DIR / "data" / "bronze" / "teams"
SILVER_RAW_DIR = BASE_DIR / "data" / "silver" / "teams" / "csv_raw"
SILVER_CLEAN_DIR = BASE_DIR / "data" / "silver" / "teams" / "csv_clean"

TEAM_FILES = [
    "teams_attacking",
    "teams_defending",
    "teams_passing",
    "teams_pressing",
    "teams_sequences",
    "teams_misc",
]

print(f"Project root : {BASE_DIR}")
print(f"Source dir   : {SOURCE_DIR}")
print(f"Bronze dir   : {BRONZE_DIR}")
print(f"Silver raw   : {SILVER_RAW_DIR}")
print(f"Silver clean : {SILVER_CLEAN_DIR}")

---
## 🥉 Bronze Layer — Ingestion

Menyalin 6 file JSON dari `datasets/teams/` (Source) ke `data/bronze/teams/` (Bronze).

Tujuannya: data source **tidak tersentuh**, kita kerja dari salinannya.

In [ ]:
# Pastikan folder tujuan ada
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

for name in TEAM_FILES:
    src = SOURCE_DIR / f"{name}.json"
    dst = BRONZE_DIR / f"{name}.json"
    
    if not src.exists():
        print(f"⚠️  SKIP — tidak ditemukan: {src.name}")
        continue
    
    shutil.copy2(src, dst)
    print(f"✅  {src.name}  →  bronze/{dst.name}")

print(f"\nFile di bronze: {list(BRONZE_DIR.glob('*.json'))}")

### 🔍 Cek isi salah satu file Bronze

In [ ]:
# Baca salah satu JSON dari Bronze untuk cek isinya
with open(BRONZE_DIR / "teams_attacking.json", "r") as f:
    data = json.load(f)

print(f"Jumlah tim: {len(data)}")
print(f"Contoh record pertama:")
data[0]

---
## 🥈 Silver Layer — Step 1: JSON → CSV Raw (csv_0)

Konversi setiap JSON dari Bronze ke CSV **apa adanya**, tanpa mengubah isi.

Ini adalah "checkpoint" — kalau ada bug di transformasi, kita bisa cek dari sini.

In [ ]:
SILVER_RAW_DIR.mkdir(parents=True, exist_ok=True)

for name in TEAM_FILES:
    src = BRONZE_DIR / f"{name}.json"
    dst = SILVER_RAW_DIR / f"{name}.csv"
    
    if not src.exists():
        print(f"⚠️  SKIP — tidak ditemukan: {src.name}")
        continue
    
    df = pd.read_json(src)
    df.to_csv(dst, index=False)
    print(f"✅  {name}.json  →  csv_raw/{name}.csv  ({len(df)} rows, {len(df.columns)} cols)")

print("\nStep 1 selesai.")

### 🔍 Cek CSV Raw — Perhatikan kolom % masih berupa string

In [ ]:
# Lihat CSV raw — perhatikan conversion_pct masih "13.73%"
df_raw = pd.read_csv(SILVER_RAW_DIR / "teams_attacking.csv")
print("=== teams_attacking (RAW) ===")
print(f"Tipe data conversion_pct: {df_raw['conversion_pct'].dtype}")
print(f"Contoh value: {df_raw['conversion_pct'].iloc[0]}")
print()
df_raw.head()

In [ ]:
# Cek semua 6 CSV raw — ringkasan
for name in TEAM_FILES:
    df = pd.read_csv(SILVER_RAW_DIR / f"{name}.csv")
    print(f"{name:25s} → {len(df):3d} rows, {len(df.columns):2d} cols, dtypes: {dict(df.dtypes.value_counts())}")

---
## 🥈 Silver Layer — Step 2: CSV Raw → CSV Clean (csv_1)

Transformasi utama:
- Menghapus simbol `%` dari kolom persentase
- Konversi dari string ke `float`

Contoh: `"13.73%"` → `13.73`

In [ ]:
def strip_pct(series: pd.Series) -> pd.Series:
    """Hapus simbol '%' dan konversi ke float. Contoh: '13.73%' → 13.73"""
    return series.str.replace("%", "", regex=False).astype(float)


def clean_attacking(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["conversion_pct"] = strip_pct(df["conversion_pct"])
    return df


def clean_defending(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["avg_possession_pct", "ground_duels_won_pct", "aerial_duels_won_pct"]:
        df[col] = strip_pct(df[col])
    return df


def clean_passing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pct_cols = [
        "avg_possession_pct", "passes_pct", "final_third_pct",
        "direction_fwd_pct", "direction_bwd_pct",
        "direction_left_pct", "direction_right_pct",
        "crosses_pct",
    ]
    for col in pct_cols:
        df[col] = strip_pct(df[col])
    return df


def clean_pressing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["high_turnovers_shot_pct"] = strip_pct(df["high_turnovers_shot_pct"])
    return df


def clean_sequences(df: pd.DataFrame) -> pd.DataFrame:
    """Semua kolom sudah numerik, tidak perlu perubahan."""
    return df.copy()


def clean_misc(df: pd.DataFrame) -> pd.DataFrame:
    """Semua kolom sudah integer, tidak perlu perubahan."""
    return df.copy()


# Mapping nama file → fungsi pembersihan
CLEANERS = {
    "teams_attacking": clean_attacking,
    "teams_defending": clean_defending,
    "teams_passing": clean_passing,
    "teams_pressing": clean_pressing,
    "teams_sequences": clean_sequences,
    "teams_misc": clean_misc,
}

print("Fungsi cleaner sudah didefinisikan ✅")

In [ ]:
SILVER_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name in TEAM_FILES:
    src = SILVER_RAW_DIR / f"{name}.csv"
    dst = SILVER_CLEAN_DIR / f"{name}.csv"
    
    if not src.exists():
        print(f"⚠️  SKIP — tidak ditemukan: {src.name}")
        continue
    
    df = pd.read_csv(src)
    cleaner = CLEANERS.get(name)
    if cleaner:
        df = cleaner(df)
    
    df.to_csv(dst, index=False)
    print(f"✅  {name} (raw → clean)  ({len(df)} rows, {len(df.columns)} cols)")

print("\nStep 2 selesai.")

### 🔍 Cek CSV Clean — Kolom % sekarang sudah jadi float

In [ ]:
# Bandingkan RAW vs CLEAN
df_raw = pd.read_csv(SILVER_RAW_DIR / "teams_attacking.csv")
df_clean = pd.read_csv(SILVER_CLEAN_DIR / "teams_attacking.csv")

print("=== PERBANDINGAN: teams_attacking ===")
print(f"\nRAW   → conversion_pct dtype: {df_raw['conversion_pct'].dtype}, contoh: {df_raw['conversion_pct'].iloc[0]}")
print(f"CLEAN → conversion_pct dtype: {df_clean['conversion_pct'].dtype}, contoh: {df_clean['conversion_pct'].iloc[0]}")

In [ ]:
# Lihat seluruh data clean — teams_attacking
df_clean

---
## 📊 Eksplorasi Data — Semua 6 Tabel Clean

In [ ]:
# Load semua 6 CSV clean ke dictionary of DataFrames
dfs = {}
for name in TEAM_FILES:
    dfs[name] = pd.read_csv(SILVER_CLEAN_DIR / f"{name}.csv")
    print(f"{name:25s} → {len(dfs[name]):3d} rows, {len(dfs[name].columns):2d} cols")

print(f"\nTotal tabel: {len(dfs)}")

In [ ]:
# Cek tipe data semua kolom per tabel
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"📋 {name}")
    print(f"{'='*60}")
    print(df.dtypes)
    print()

In [ ]:
# Lihat tabel satu per satu — ganti nama di bawah untuk eksplorasi
# Pilihan: teams_attacking, teams_defending, teams_passing, 
#          teams_pressing, teams_sequences, teams_misc

tabel = "teams_attacking"  # ← ganti di sini
dfs[tabel]

In [ ]:
# Statistik deskriptif
dfs[tabel].describe()

---
## ✅ Ringkasan Pipeline

| Layer | Lokasi | Format | Isi |
|---|---|---|---|
| **Source** | `datasets/teams/` | 6 JSON | Data asli, tidak tersentuh |
| **Bronze** | `data/bronze/teams/` | 6 JSON | Salinan data mentah |
| **Silver (csv_raw)** | `data/silver/teams/csv_raw/` | 6 CSV | Konversi JSON → CSV, apa adanya |
| **Silver (csv_clean)** | `data/silver/teams/csv_clean/` | 6 CSV | Bersih — `%` dihapus, siap load ke PostgreSQL |